# Combinatorial CoWork 2026 — Notebook 06: Featurization and repeated computation

Audience:
- Participants who want one notebook showing that the library is not limited to one-off algebraic queries: it can also support repeated featurization and lightweight experiment packaging.

Learning goals:
- Build a small labeled family of synthetic rectangle modules.
- Common-encode the family once so that downstream feature extraction is uniform.
- Use a composite featurizer to produce one feature matrix for several samples at once.
- Reuse the same `SessionCache()` across repeated passes and compare timings.
- Package the same workflow as an experiment that writes reusable feature tables and metadata.


## Outline

1. Build a small labeled synthetic family
2. Common-encode the family once and inspect one representative geometry
3. Declare a reusable featurization pipeline
4. Compute one feature matrix and a simple classwise statistical summary
5. Repeat the computation with a shared `SessionCache()` and an in-place buffer
6. Package the same workflow as an experiment and export the results


In [1]:
NOTEBOOK_STEM = "06_featurization_and_repeated_computation"

_TO_ROOT = let
    dir = abspath(pwd())
    root = nothing
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "TamerOp.jl"))
            root = dir
            break
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repo root containing Project.toml and src/TamerOp.jl from pwd()=$(pwd()).")
        dir = parent
    end
    root
end

import Pkg
Pkg.activate(_TO_ROOT; io=devnull)

if !isdefined(Main, :TamerOp)
    @eval Main using TamerOp
end

TO = Main.TamerOp
FEA = TO.Featurizers
CM = TO.CoreModules
OPT = TO.Options
SD = TO.SyntheticData

using Statistics

sc = TO.SessionCache()
outdir = joinpath(_TO_ROOT, "examples", "_outputs", "combinatorial_cowork_2026", NOTEBOOK_STEM)
mkpath(outdir)


"/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation"

## 1. Build a small labeled family

We use a tiny family of rectangle-indicator flanges. The first three samples have one bar each; the last three have two bars each. That gives us a toy labeled dataset without leaving the synthetic-data surface.


In [2]:
family_bars = [
    [([0, 0], [2, 2])],
    [([0, 0], [2, 3])],
    [([1, 0], [3, 2])],
    [([0, 0], [2, 2]), ([1, 1], [3, 3])],
    [([0, 0], [2, 3]), ([1, 1], [3, 4])],
    [([1, 0], [3, 2]), ([2, 1], [4, 3])],
]

family_ids = ["single_1", "single_2", "single_3", "double_1", "double_2", "double_3"]
family_labels = ["single", "single", "single", "double", "double", "double"]

sources = [
    SD.orthant_bar_flange(bars=bars, field=CM.QQField())
    for bars in family_bars
]

enc_opts = OPT.EncodingOptions(backend=:zn, max_regions=100_000, poset_kind=:signature)

(; nsamples=length(sources),
   family_ids,
   family_labels,
   first_source=TO.describe(sources[1]),
   enc_opts)


(nsamples = 6, family_ids = ["single_1", "single_2", "single_3", "double_1", "double_2", "double_3"], family_labels = ["single", "single", "single", "double", "double", "double"], first_source = (kind = :flange, ambient_dim = 2, field = "TamerOp.CoreModules.CoeffFields.QQField", nflats = 1, ninjectives = 1, matrix_size = (1, 1)), enc_opts = EncodingOptions(backend=:zn, poset_kind=:signature, max_regions=100000))

## 2. Common-encode the family once

For repeated computation we want one shared encoding, not six unrelated ones. `encode(sources, enc_opts; ...)` gives us a vector of `EncodingResult`s on one common finite poset and one common classifier map.

We also create identity-based lookup tables for sample ids and labels. That keeps the later `FeatureSet` outputs readable without teaching any extra wrapper types.


In [3]:
enc_samples = TO.encode(sources, enc_opts; cache=:auto)

idmap = IdDict(enc_samples[i] => family_ids[i] for i in eachindex(enc_samples))
labelmap = IdDict(enc_samples[i] => family_labels[i] for i in eachindex(enc_samples))

regions_spec = TO.Visualization.visual_spec(TO.encoding_map(enc_samples[1]); kind=:regions)

(; representative_encoding=TO.describe(enc_samples[1]),
   shared_poset=all(enc -> TO.encoding_poset(enc) === TO.encoding_poset(enc_samples[1]), enc_samples),
   shared_classifier=all(enc -> TO.encoding_map(enc) === TO.encoding_map(enc_samples[1]), enc_samples),
   representative_regions=TO.Visualization.visual_summary(regions_spec))


(representative_encoding = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :zn, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), shared_poset = true, shared_classifier = true, representative_regions = (kind = :visualization_spec, visual_kind = :regions, title = "Zn encoding", subtitle = "", nlayers = 1, npanels = 0, layer_types = (:RectLayer,), axes = (xlabel = "g1", ylabel = "g2", xlimits = (-1.0, 5.0), ylimits = (-1.0, 5.0), zlabel = "z", zlimits 

## 3. Declare a reusable featurization pipeline

The statistics pipeline will combine two different feature families:

- an `EulerSurfaceSpec`, which samples a 2-parameter Euler-style summary on a fixed grid,
- an `MPLandscapeSpec`, which samples a multiparameter landscape along one slice direction.

We package them together as one `CompositeSpec` so that one featurization call produces one matrix with a stable column order.


In [ ]:
euler_spec = TO.EulerSurfaceSpec(
    axes=([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0]),
    axes_policy=:as_given,
    strict=false,
)

land_spec = TO.MPLandscapeSpec(
    directions=[[1.0, 1.0]],
    offsets=[[0.0, 0.0]],
    tgrid=collect(0.0:0.5:4.0),
    kmax=2,
    strict=false,
)

stat_spec = TO.CompositeSpec((euler_spec, land_spec), namespacing=true)
batch = TO.BatchOptions(threaded=false, backend=:serial, deterministic=true)


BatchOptions(false, :serial, false, true, 0)

`describe(...)` is still the generic inspection surface. The owner-local summary and validator are useful here because this notebook is spending all of its time inside the featurization owner.


In [5]:
spec_summary = TO.describe(stat_spec)
spec_owner_summary = FEA.featurizer_summary(stat_spec)
spec_check = FEA.check_featurizer_spec(stat_spec)

(; spec_summary, spec_owner_summary, spec_check, batch)


(spec_summary = (kind = :featurizer_spec, spec_type = :CompositeSpec, nfeatures = 54, feature_axes = (namespacing = true, components = Any[(name = "EulerSurfaceSpec", spec_type = "EulerSurfaceSpec", start = 1, stop = 36, axes = (axis_1 = [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], axis_2 = [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], axes_policy = :as_given)), (name = "MPLandscapeSpec", spec_type = "MPLandscapeSpec", start = 37, stop = 54, axes = (direction = [1], offset = [1], k = [1, 2], t = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0], directions = [[1.0, 1.0]], offsets = [[0.0, 0.0]]))]), fields = (specs = (EulerSurfaceSpec(nfeatures=36), MPLandscapeSpec(nfeatures=18)), namespacing = true)), spec_owner_summary = (kind = :featurizer_spec, spec_type = :CompositeSpec, nfeatures = 54, feature_axes = (namespacing = true, components = Any[(name = "EulerSurfaceSpec", spec_type = "EulerSurfaceSpec", start = 1, stop = 36, axes = (axis_1 = [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], axis_2 = [-1.0, 0.0, 1.0, 2.0, 3.0, 4.0],

## 4. Compute one feature matrix

`featurize(...)` is the dataset-level entrypoint. The output is a `FeatureSet`, which carries the matrix itself, the feature names, the sample ids, and any labels we attached through `idfun` and `labelfun`.

After the first pass, we compute a very simple classwise summary: the mean feature vector for the single-bar samples and the mean feature vector for the double-bar samples.


In [6]:
fs = TO.featurize(
    enc_samples,
    stat_spec;
    batch=batch,
    cache=:auto,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)

X = FEA.feature_matrix(fs)
labels = FEA.sample_labels(fs)
single_idx = findall(==("single"), labels)
double_idx = findall(==("double"), labels)

single_mean = vec(mean(X[single_idx, :]; dims=1))
double_mean = vec(mean(X[double_idx, :]; dims=1))
preview_n = min(10, size(X, 2))

feature_preview = (
    feature_names = TO.feature_names(fs)[1:preview_n],
    single_mean = round.(single_mean[1:preview_n]; digits=3),
    double_mean = round.(double_mean[1:preview_n]; digits=3),
)

(; feature_set=FEA.feature_set_summary(fs), feature_preview)


(feature_set = (kind = :feature_set, nsamples = 6, nfeatures = 54, feature_names = [:EulerSurfaceSpec__euler__f1, :EulerSurfaceSpec__euler__f2, :EulerSurfaceSpec__euler__f3, :EulerSurfaceSpec__euler__f4, :EulerSurfaceSpec__euler__f5, :EulerSurfaceSpec__euler__f6, :EulerSurfaceSpec__euler__f7, :EulerSurfaceSpec__euler__f8, :EulerSurfaceSpec__euler__f9, :EulerSurfaceSpec__euler__f10  …  :MPLandscapeSpec__mp_landscape__d1_o1_k1_t5, :MPLandscapeSpec__mp_landscape__d1_o1_k2_t5, :MPLandscapeSpec__mp_landscape__d1_o1_k1_t6, :MPLandscapeSpec__mp_landscape__d1_o1_k2_t6, :MPLandscapeSpec__mp_landscape__d1_o1_k1_t7, :MPLandscapeSpec__mp_landscape__d1_o1_k2_t7, :MPLandscapeSpec__mp_landscape__d1_o1_k1_t8, :MPLandscapeSpec__mp_landscape__d1_o1_k2_t8, :MPLandscapeSpec__mp_landscape__d1_o1_k1_t9, :MPLandscapeSpec__mp_landscape__d1_o1_k2_t9], feature_axes = (namespacing = true, components = Any[(name = "EulerSurfaceSpec", spec_type = "EulerSurfaceSpec", start = 1, stop = 36, axes = (axis_1 = [-1.0, 0.

## 5. Repeat the same computation with a shared cache

The main performance point of this notebook is that the same feature pipeline can be run several times on the same encoded family without rebuilding every intermediate object.

We warm the `SessionCache()` once, then measure:

- one `cache=:auto` pass,
- two cached `featurize(...)` passes,
- one cached `batch_transform!(...)` pass into a preallocated matrix.

This is not a benchmark harness. It is a notebook-scale timing sanity check after compilation has already happened.


In [7]:
TO.featurize(
    enc_samples,
    stat_spec;
    batch=batch,
    cache=sc,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)

t0 = time_ns()
fs_auto = TO.featurize(
    enc_samples,
    stat_spec;
    batch=batch,
    cache=:auto,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)
elapsed_auto = (time_ns() - t0) / 1.0e9

t1 = time_ns()
fs_cache_1 = TO.featurize(
    enc_samples,
    stat_spec;
    batch=batch,
    cache=sc,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)
elapsed_cache_1 = (time_ns() - t1) / 1.0e9

t2 = time_ns()
fs_cache_2 = TO.featurize(
    enc_samples,
    stat_spec;
    batch=batch,
    cache=sc,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)
elapsed_cache_2 = (time_ns() - t2) / 1.0e9

Xbuf = Matrix{Float64}(undef, size(FEA.feature_matrix(fs_cache_2), 1), size(FEA.feature_matrix(fs_cache_2), 2))

t3 = time_ns()
fs_bang = TO.batch_transform!(
    Xbuf,
    enc_samples,
    stat_spec;
    batch=batch,
    cache=sc,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
)
elapsed_bang = (time_ns() - t3) / 1.0e9

(; elapsed_auto,
   elapsed_cache_1,
   elapsed_cache_2,
   elapsed_bang,
   same_cached = FEA.feature_matrix(fs_cache_1) == FEA.feature_matrix(fs_cache_2),
   same_inplace = FEA.feature_matrix(fs_bang) == FEA.feature_matrix(fs_cache_2),
   sample_ids = FEA.sample_ids(fs_cache_2),
   sample_labels = FEA.sample_labels(fs_cache_2))


(elapsed_auto = 0.44368172, elapsed_cache_1 = 0.475651849, elapsed_cache_2 = 0.433218657, elapsed_bang = 0.49569633, same_cached = true, same_inplace = true, sample_ids = ["single_1", "single_2", "single_3", "double_1", "double_2", "double_3"], sample_labels = ["single", "single", "single", "double", "double", "double"])

## 6. Package the same workflow as an experiment

For a repeatable statistics pipeline, the next step is not another ad hoc notebook cell. It is an `ExperimentSpec` that fixes:

- the featurizers,
- the batching policy,
- the cache contract,
- the sample ids/labels,
- and the output policy.

Here we write lightweight CSV tables and JSON metadata into this notebook's output directory.


In [8]:
exp_io = FEA.ExperimentIOConfig(
    outdir=outdir,
    prefix="cowork06",
    formats=(:csv_wide,),
    write_metadata=true,
    overwrite=true,
)

exp = FEA.ExperimentSpec(
    (euler_spec, land_spec);
    name="cowork06_stats_pipeline",
    batch=batch,
    cache=sc,
    idfun=s -> idmap[s],
    labelfun=s -> labelmap[s],
    io=exp_io,
    metadata=(suite="combinatorial_cowork_2026", notebook=NOTEBOOK_STEM, family="common_encoded_rectangle_flanges"),
)

(; experiment=FEA.experiment_summary(exp), io=TO.describe(exp_io))


(experiment = (kind = :experiment_spec, name = "cowork06_stats_pipeline", nfeaturizers = 2, featurizer_types = [:EulerSurfaceSpec, :MPLandscapeSpec], on_unsupported = :error, cache = SessionCache(encoding=0, modules=0, zn=0), batch = BatchOptions(false, :serial, false, true, 0), io = (kind = :experiment_io_config, outdir = "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation", prefix = "cowork06", format = :wide, formats = [:csv_wide], write_metadata = true, overwrite = true), metadata_keys = [:suite, :notebook, :family]), io = (kind = :experiment_io_config, outdir = "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation", prefix = "cowork06", format = :wide, formats = [:csv_wide], write_metadata = true, overwrite = true))

The first experiment run fills and records the pipeline artifacts. The second run reuses the same session cache. That second timing is the clearest demonstration in this notebook that repeated computation is a first-class use case.


In [9]:
exp_run_1 = TO.run_experiment(exp, enc_samples)
exp_run_2 = TO.run_experiment(exp, enc_samples)

run1 = TO.describe(exp_run_1)
run2 = TO.describe(exp_run_2)
artifacts2 = map(TO.describe, FEA.artifacts(exp_run_2))
speedup = run2.total_elapsed_seconds == 0.0 ? Inf : run1.total_elapsed_seconds / run2.total_elapsed_seconds
artifact_timing = Dict(String(d.key) => d.elapsed_seconds for d in artifacts2)

(; first_run_seconds = round(run1.total_elapsed_seconds; digits=3),
   second_run_seconds = round(run2.total_elapsed_seconds; digits=6),
   speedup = round(speedup; digits=1),
   artifact_timing,
   second_run_artifacts = artifacts2)


(first_run_seconds = 4.805, second_run_seconds = 0.006829, speedup = 703.7, artifact_timing = Dict("s02__mplandscapespec" => 0.000891371, "s01__eulersurfacespec" => 0.000316089), second_run_artifacts = @NamedTuple{kind::Symbol, key::Symbol, spec_type::Symbol, nsamples::Int64, nfeatures::Int64, elapsed_seconds::Float64, feature_paths::Dict{Symbol, String}, metadata_path::String, has_features::Bool}[(kind = :experiment_artifact, key = :s01__eulersurfacespec, spec_type = :EulerSurfaceSpec, nsamples = 6, nfeatures = 36, elapsed_seconds = 0.000316089, feature_paths = Dict(:csv_wide => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/cowork06__cowork06_stats_pipeline/s01__eulersurfacespec__wide.csv"), metadata_path = "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/cowork06__cowork06_stats_pipeline/s01__eulersurfacespe

## 7. Export one visual and the experiment artifacts

The feature tables are already written by `run_experiment(...)`. We also export one representative encoding visual so the notebook leaves behind both a figure and machine-readable feature artifacts.


In [10]:
exports = TO.save_visuals(
    outdir,
    [
        (; stem="representative_regions", obj=TO.encoding_map(enc_samples[1]), kind=:regions),
    ];
    format=:html,
)

artifact_paths = Dict(
    String(d.key) => d.feature_paths
    for d in map(TO.describe, FEA.artifacts(exp_run_2))
)

(; visual_paths = Dict(TO.export_stem(r) => TO.export_path(r) for r in exports),
   experiment_manifest = FEA.manifest_path(exp_run_2),
   artifact_paths)


(visual_paths = Dict("representative_regions" => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/representative_regions.html"), experiment_manifest = "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/cowork06__cowork06_stats_pipeline/manifest.json", artifact_paths = Dict("s02__mplandscapespec" => Dict(:csv_wide => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/cowork06__cowork06_stats_pipeline/s02__mplandscapespec__wide.csv"), "s01__eulersurfacespec" => Dict(:csv_wide => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/06_featurization_and_repeated_computation/cowork06__cowork06_stats_pipeline/s01__eulersurfacespec__wide.csv")))

## Pitfall and extension

Common pitfall:
- Comparing the first ever notebook cell execution against a warm cached run mixes compilation cost with runtime cost. This notebook avoids that by warming the session cache before the timing cell.

Useful extension:
- Add a second landscape direction such as `[1.0, 0.0]`, rerun Sections 4-6, and compare how the feature matrix width and the second-run timings change.


In [11]:
# Exercise scaffold:
# 1. Add a second direction to `land_spec`.
# 2. Rebuild `stat_spec`.
# 3. Rerun the feature-matrix, timing, and experiment cells.
# 4. Compare the first ten class-mean features before and after.